# 02 - Assessing differential abundance

In [ ]:
%autosave 0

## Load libraries

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import pertpy as pt
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from dynaconf import Dynaconf
from tqdm import tqdm
import marsilea as ma
import marsilea.plotter as mp

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Utility paremeters

In [ ]:
# Specify color palette for each comparison
comparison_palette = {
    "non_eecs_annotation": {
        "Stem Cells": "#1f77b4",          # deep blue
        "TA Cells": "#4c91c6",            # medium blue

        # absorptive lineage
        "Enterocytes": "#2ca02c",         # green

        # secretory lineage
        "Goblet Cells": "#ff7f0e",        # orange
    },
    "eecs_annotation": {
        # EEC progenitors
        "EEC Progenitors": "#9467bd",   # purple
        
        # EEC subtypes 
        "EC Cells": "#d62728",            # strong red
        "X Cells": "#e377c2",             # magenta
        "D Cells": "#17becf",             # cyan / turquoise
        "I/N Cells": "#7f7f7f",           # dark grey
        "K Cells": "#bcbd22",             # olive
    },
}

In [ ]:
settings.ANALYSIS.tables_dir

## Load cell proportions per condition

In [ ]:
df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "all_eecs" / "eecs_annotation_control_cell_prop_per_sample.csv", index_col=0) 
# Subset to non-empty columns i.e. only Controls
df = df.dropna(axis=1, how='all')
# Pool controls for each day i.e. Cnt2_1, Cnt2_2 -> Cnt2
df.columns = df.columns.str.replace(r'(_\d+)$', '', regex=True)
# Average proportions across control replicates for each day
df = df.groupby(df.columns, axis=1).mean()

# Plot as stacked barplot using marsilea
h = ma.ClusterBoard(df, width=3, height=1.5)
h.add_layer(mp.StackBar(df, label="Cell Type", colors=comparison_palette["eecs_annotation"]))
h.add_bottom(mp.Labels(df.columns), size=1, pad=0.1)
h.add_legends()

with plt.rc_context({'axes.grid': False}):
    h.render()
    plt.savefig(sc.settings.figdir / "all_eecs_control_cell_proportions.pdf", bbox_inches='tight')

In [ ]:
df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "all_non_eecs" / "non_eecs_annotation_control_cell_prop_per_sample.csv", index_col=0) 
# Subset to non-empty columns i.e. only Controls
df = df.dropna(axis=1, how='all')
# Pool controls for each day i.e. Cnt2_1, Cnt2_2 -> Cnt2
df.columns = df.columns.str.replace(r'(_\d+)$', '', regex=True)
# Average proportions across control replicates for each day
df = df.groupby(df.columns, axis=1).mean()

# Plot as stacked barplot using marsilea
h = ma.ClusterBoard(df, width=3, height=1.5)
h.add_layer(mp.StackBar(df, label="Cell Type", colors=comparison_palette["non_eecs_annotation"]))
h.add_bottom(mp.Labels(df.columns), size=1, pad=0.1)
h.add_legends()

with plt.rc_context({'axes.grid': False}):
    h.render()
    plt.savefig(sc.settings.figdir / "all_non_eecs_control_cell_proportions.pdf", bbox_inches='tight')

In [ ]:
sc.settings.figdir